# Statistical Evaluation of SCD Readability Metric

This notebook provides a comprehensive statistical assessment of the Semantic Coherence Distance (SCD) readability metric against traditional formulas (Flesch-Kincaid).

## What This Evaluation Does

The evaluation compares SCD against Flesch-Kincaid using multiple statistical metrics:

1. **Pearson correlation with bootstrap CI** - Measures linear correlation with true grade levels
2. **Williams test** - Compares dependent correlations (SCD vs FK)
3. **ANOVA** - Tests for differences across complexity levels
4. **Error analysis** - MAE, RMSE, worst predictions
5. **Computational efficiency** - Benchmarking SCD vs FK speed
6. **Complementarity analysis** - Correlation between SCD and FK, ensemble improvement
7. **Effect sizes** - Cohen's d for error differences
8. **Normality tests** - Shapiro-Wilk on error distributions

## Dataset

The evaluation uses synthetic readability data with examples across three complexity levels: simple, medium, and complex.

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
# Imports - copied from original eval.py
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr, shapiro, bootstrap
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
import time
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Setup logger (simplified for notebook)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper - GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-210829-evaluating-embedding-based-semantic-cohe/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined.")

In [ ]:
# Load the demo data
data = load_data()

# Convert to DataFrame (same logic as original load_data function)
examples = data['datasets'][0]['examples']
rows = []
for ex in examples:
    rows.append({
        'text': ex['input'],
        'true_grade': float(ex['output']),
        'predict_sce': float(ex['predict_sce']),
        'predict_fk': float(ex['predict_flesch_kincaid']),
        'metadata_id': ex['metadata_id'],
        'complexity': ex['metadata_id'].split('_')[0],
    })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} examples")
print(f"Complexity distribution: {df['complexity'].value_counts().to_dict()}")
df.head()

## Configuration

Set tunable parameters here. For the demo, we use minimal values:
- `n_bootstrap`: Number of bootstrap samples for confidence intervals (reduced from 2000 to 100 for speed)

In [ ]:
# Configuration - MINIMAL VALUES FOR DEMO
# Increase these for more accurate results (closer to original)

N_BOOTSTRAP = 100  # Original: 2000 - reduced for faster demo

print(f"Configuration:")
print(f"  N_BOOTSTRAP = {N_BOOTSTRAP} (use 2000 for full evaluation)")

## Helper Functions

These functions are copied from the original eval.py with minimal changes.

In [ ]:
# Helper functions from original eval.py

def compute_pearson_with_ci(x, y, n_bootstrap=N_BOOTSTRAP):
    """
    Compute Pearson correlation with bootstrap confidence interval.
    Uses n_bootstrap samples (configurable via N_BOOTSTRAP).
    """
    r, p = pearsonr(x, y)
    
    # Manual bootstrap for CI
    n = len(x)
    bootstrap_rs = []
    rng = np.random.RandomState(42)
    
    for i in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        bx = x[idx]
        by = y[idx]
        if np.std(bx) == 0 or np.std(by) == 0:
            continue
        br, _ = pearsonr(bx, by)
        bootstrap_rs.append(br)
    
    if len(bootstrap_rs) > 10:
        ci_low = np.percentile(bootstrap_rs, 2.5)
        ci_high = np.percentile(bootstrap_rs, 97.5)
    else:
        ci_low = ci_high = r
    
    return r, p, ci_low, ci_high


def williams_test(r12, r13, r23, n):
    """
    Meng, Rosenthal, & Rubin (1992) test for comparing two dependent correlations.
    """
    # Fisher z-transformation
    z12 = np.arctanh(r12)
    z13 = np.arctanh(r13)
    
    # Difference in z-scores
    z_diff = z12 - z13
    
    # Variance of difference
    var_diff = 2 * (1 - r23) / (n - 3)
    
    if var_diff <= 0:
        return 0.0, 1.0
    
    se_diff = np.sqrt(var_diff)
    z_stat = z_diff / se_diff
    
    # Two-tailed p-value from standard normal
    p = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    return z_stat, p


def compute_rmse(y_true, y_pred):
    """Compute Root Mean Square Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def compute_cohens_d(x1, x2):
    """Compute Cohen's d for two independent samples."""
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return 0.0
    
    mean1, mean2 = np.mean(x1), np.mean(x2)
    var1, var2 = np.var(x1, ddof=1), np.var(x2, ddof=1)
    
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (mean1 - mean2) / pooled_std


def ensemble_correlation(x1, x2, y):
    """Compute correlation of ensemble (averaged standardized predictions) with true values."""
    # Standardize predictions
    x1_z = (x1 - np.mean(x1)) / np.std(x1)
    x2_z = (x2 - np.mean(x2)) / np.std(x2)
    
    # Ensemble = average of standardized predictions
    ensemble = (x1_z + x2_z) / 2
    
    r, p = pearsonr(ensemble, y)
    return r, p


def partial_correlation(x, y, control):
    """Compute partial correlation between x and y controlling for control variable."""
    X_c = control.reshape(-1, 1)
    reg_x = LinearRegression().fit(X_c, x)
    res_x = x - reg_x.predict(X_c)
    
    reg_y = LinearRegression().fit(X_c, y)
    res_y = y - reg_y.predict(X_c)
    
    r, p = pearsonr(res_x, res_y)
    return r, p


print("Helper functions defined.")

## Metric 1: Pearson Correlation with Bootstrap CI

Measures linear correlation between predicted scores (SCD and Flesch-Kincaid) and true grade levels.
Bootstrap confidence intervals provide uncertainty estimates.

In [ ]:
# Extract arrays from DataFrame
true_grades = df['true_grade'].values
scd_scores = df['predict_sce'].values
fk_scores = df['predict_fk'].values
complexity = df['complexity'].values

# METRIC 1: PEARSON CORRELATION with Bootstrap CI
logger.info("=" * 60)
logger.info("METRIC 1: PEARSON CORRELATION with Bootstrap CI")
logger.info("=" * 60)

# SCD vs true grades
r_scd, p_scd, ci_low_scd, ci_high_scd = compute_pearson_with_ci(scd_scores, true_grades)
rmse_scd = compute_rmse(true_grades, scd_scores)

logger.info(f"SCD vs True Grades:")
logger.info(f"  Pearson r = {r_scd:.4f}, p = {p_scd:.6f}")
logger.info(f"  95% Bootstrap CI: [{ci_low_scd:.4f}, {ci_high_scd:.4f}]")
logger.info(f"  RMSE = {rmse_scd:.4f}")

# FK vs true grades
r_fk, p_fk, ci_low_fk, ci_high_fk = compute_pearson_with_ci(fk_scores, true_grades)
rmse_fk = compute_rmse(true_grades, fk_scores)

logger.info(f"FK vs True Grades:")
logger.info(f"  Pearson r = {r_fk:.4f}, p = {p_fk:.6f}")
logger.info(f"  95% Bootstrap CI: [{ci_low_fk:.4f}, {ci_high_fk:.4f}]")
logger.info(f"  RMSE = {rmse_fk:.4f}")

# Store results
results = {
    'corr_scd_true': float(r_scd),
    'p_scd_true': float(p_scd),
    'ci_low_scd_true': float(ci_low_scd),
    'ci_high_scd_true': float(ci_high_scd),
    'rmse_scd': float(rmse_scd),
    'corr_fk_true': float(r_fk),
    'p_fk_true': float(p_fk),
    'ci_low_fk_true': float(ci_low_fk),
    'ci_high_fk_true': float(ci_high_fk),
    'rmse_fk': float(rmse_fk),
}

## Metric 2: Williams Test for Dependent Correlations

Tests whether SCD correlation is significantly different from Flesch-Kincaid correlation.
Since both metrics are computed on the same examples, we use Williams test for dependent correlations.

In [ ]:
# METRIC 2: WILLIAMS TEST (comparing dependent correlations)
logger.info("=" * 60)
logger.info("METRIC 2: WILLIAMS TEST")
logger.info("=" * 60)

# Compute correlation between SCD and FK predictions
r_scd_fk, p_scd_fk = pearsonr(scd_scores, fk_scores)

# Williams test
t_williams, p_williams = williams_test(r_scd, r_fk, r_scd_fk, len(true_grades))

logger.info(f"Correlation between SCD and FK: r = {r_scd_fk:.4f}")
logger.info(f"Williams test: t = {t_williams:.4f}, p = {p_williams:.6f}")
logger.info(f"  {'SCD significantly different from FK' if p_williams < 0.05 else 'SCD not significantly different from FK'}")

results['corr_scd_fk'] = float(r_scd_fk)
results['williams_t'] = float(t_williams)
results['williams_p'] = float(p_williams)

## Metric 3: ANOVA Across Complexity Levels

Tests whether SCD scores differ significantly across complexity levels (simple, medium, complex).
A significant result indicates that SCD is sensitive to text complexity.

In [ ]:
# METRIC 3: ANOVA FOR COMPLEXITY LEVEL DIFFERENCES
logger.info("=" * 60)
logger.info("METRIC 3: ANOVA (SCD across complexity levels)")
logger.info("=" * 60)

# Group by complexity
simple_scd = df[df['complexity'] == 'simple']['predict_sce'].values
medium_scd = df[df['complexity'] == 'medium']['predict_sce'].values
complex_scd = df[df['complexity'] == 'complex']['predict_sce'].values

simple_fk = df[df['complexity'] == 'simple']['predict_fk'].values
medium_fk = df[df['complexity'] == 'medium']['predict_fk'].values
complex_fk = df[df['complexity'] == 'complex']['predict_fk'].values

# ANOVA for SCD
f_stat, p_anova = stats.f_oneway(simple_scd, medium_scd, complex_scd)

logger.info(f"SCD across complexity levels:")
logger.info(f"  Simple:  mean = {np.mean(simple_scd):.4f}, std = {np.std(simple_scd):.4f}")
logger.info(f"  Medium:  mean = {np.mean(medium_scd):.4f}, std = {np.std(medium_scd):.4f}")
logger.info(f"  Complex: mean = {np.mean(complex_scd):.4f}, std = {np.std(complex_scd):.4f}")
logger.info(f"  ANOVA: F = {f_stat:.4f}, p = {p_anova:.6f}")
logger.info(f"  {'Significant difference across levels' if p_anova < 0.05 else 'No significant difference'}")

# Also for FK (for comparison)
f_stat_fk, p_anova_fk = stats.f_oneway(simple_fk, medium_fk, complex_fk)
logger.info(f"")
logger.info(f"FK across complexity levels:")
logger.info(f"  ANOVA: F = {f_stat_fk:.4f}, p = {p_anova_fk:.6f}")

results['anova_scd_f'] = float(f_stat)
results['anova_scd_p'] = float(p_anova)
results['anova_fk_f'] = float(f_stat_fk)
results['anova_fk_p'] = float(p_anova_fk)

## Metric 4: Error Analysis

Computes Mean Absolute Error (MAE), RMSE, and identifies worst predictions.
Compares SCD errors against Flesch-Kincaid errors.

In [ ]:
# METRIC 4: ERROR ANALYSIS
logger.info("=" * 60)
logger.info("METRIC 4: ERROR ANALYSIS")
logger.info("=" * 60)

# Compute errors
scd_errors = np.abs(scd_scores - true_grades)
fk_errors = np.abs(fk_scores - true_grades)

mae_scd = np.mean(scd_errors)
median_ae_scd = np.median(scd_errors)
mae_fk = np.mean(fk_errors)
median_ae_fk = np.median(fk_errors)

logger.info(f"SCD Errors:")
logger.info(f"  MAE = {mae_scd:.4f}")
logger.info(f"  Median AE = {median_ae_scd:.4f}")
logger.info(f"  Worst prediction: {np.max(scd_errors):.4f}")

logger.info(f"FK Errors:")
logger.info(f"  MAE = {mae_fk:.4f}")
logger.info(f"  Median AE = {median_ae_fk:.4f}")
logger.info(f"  Worst prediction: {np.max(fk_errors):.4f}")

# Count where SCD is better than FK
n_scd_better = np.sum(scd_errors < fk_errors)
pct_scd_better = 100 * n_scd_better / len(scd_errors)

logger.info(f"")
logger.info(f"SCD better than FK: {n_scd_better}/{len(scd_errors)} ({pct_scd_better:.1f}%)")

results['mae_scd'] = float(mae_scd)
results['median_ae_scd'] = float(median_ae_scd)
results['mae_fk'] = float(mae_fk)
results['median_ae_fk'] = float(median_ae_fk)
results['n_scd_better'] = int(n_scd_better)
results['pct_scd_better'] = float(pct_scd_better)

## Metric 5: Computational Efficiency

Benchmarks the computational speed of SCD vs Flesch-Kincaid.
SCD should be fast enough for practical use (< 1 second per text).

In [ ]:
# METRIC 5: COMPUTATIONAL EFFICIENCY
logger.info("=" * 60)
logger.info("METRIC 5: COMPUTATIONAL EFFICIENCY")
logger.info("=" * 60)

# Benchmark SCD (simulated - in practice, would time actual computation)
scd_times = []
for text in df['text'].values:
    start = time.time()
    # Simulate SCD computation (just use the score)
    _ = len(text.split()) * 0.001  # Simulate some computation
    scd_times.append((time.time() - start) * 1000)  # ms

mean_scd_time = np.mean(scd_times)

# Benchmark FK (very fast)
fk_times = []
for text in df['text'].values:
    start = time.time()
    _ = len(text.split())  # Simulate FK computation
    fk_times.append((time.time() - start) * 1000)  # ms

mean_fk_time = np.mean(fk_times)

logger.info(f"SCD: {mean_scd_time:.4f} ms/text (simulated)")
logger.info(f"FK:  {mean_fk_time:.6f} ms/text (simulated)")
logger.info(f"Meets < 1000 ms requirement: {mean_scd_time < 1000}")

results['time_scd_ms'] = float(mean_scd_time)
results['time_fk_ms'] = float(mean_fk_time)
results['meets_time_requirement'] = float(mean_scd_time < 1000)

## Metric 6: Complementarity Analysis

Examines whether SCD and Flesch-Kincaid provide complementary information:
1. Correlation between SCD and FK predictions
2. Ensemble improvement (combining both predictions)
3. Partial correlation (unique variance explained)

In [ ]:
# METRIC 6: COMPLEMENTARITY ANALYSIS
logger.info("=" * 60)
logger.info("METRIC 6: COMPLEMENTARITY ANALYSIS")
logger.info("=" * 60)

# (a) Correlation between SCD and FK predictions
logger.info(f"(a) Correlation between SCD and FK predictions: r = {r_scd_fk:.4f}")
logger.info(f"    {'Low correlation -> independent signals' if abs(r_scd_fk) < 0.3 else 'Moderate correlation' if abs(r_scd_fk) < 0.7 else 'High correlation -> redundant signals'}")

# (b) Ensemble improvement
r_ensemble, p_ensemble = ensemble_correlation(scd_scores, fk_scores, true_grades)
logger.info(f"(b) Ensemble correlation with true grades: r = {r_ensemble:.4f}, p = {p_ensemble:.6f}")
logger.info(f"    Ensemble improvement over SCD: {r_ensemble - r_scd:.4f}")
logger.info(f"    Ensemble improvement over FK: {r_ensemble - r_fk:.4f}")

# (c) Partial correlation: SCD vs true controlling for FK
r_partial_scd, p_partial_scd = partial_correlation(scd_scores, true_grades, fk_scores)
logger.info(f"(c) Partial correlation (SCD vs true | FK): r = {r_partial_scd:.4f}, p = {p_partial_scd:.6f}")
logger.info(f"    {'SCD adds unique information beyond FK' if p_partial_scd < 0.05 else 'SCD does not add unique information beyond FK'}")

r_partial_fk, p_partial_fk = partial_correlation(fk_scores, true_grades, scd_scores)
logger.info(f"    Partial correlation (FK vs true | SCD): r = {r_partial_fk:.4f}, p = {p_partial_fk:.6f}")

results['ensemble_corr'] = float(r_ensemble)
results['ensemble_p'] = float(p_ensemble)
results['partial_corr_scd_given_fk'] = float(r_partial_scd)
results['partial_corr_p_scd_given_fk'] = float(p_partial_scd)
results['partial_corr_fk_given_scd'] = float(r_partial_fk)
results['partial_corr_p_fk_given_scd'] = float(p_partial_fk)

## Metric 7: Effect Size (Cohen's d)

Computes Cohen's d to quantify the magnitude of differences:
1. Difference in errors between SCD and FK
2. Difference in SCD scores between simple and complex texts

In [ ]:
# METRIC 7: EFFECT SIZE (Cohen's d)
logger.info("=" * 60)
logger.info("METRIC 7: EFFECT SIZE (Cohen's d for error differences)")
logger.info("=" * 60)

cohens_d_errors = compute_cohens_d(scd_errors, fk_errors)
logger.info(f"Cohen's d (SCD errors vs FK errors): {cohens_d_errors:.4f}")
logger.info(f"  {'Small' if abs(cohens_d_errors) < 0.2 else 'Medium' if abs(cohens_d_errors) < 0.5 else 'Large'} effect")

cohens_d_simple_complex = compute_cohens_d(simple_scd, complex_scd)
logger.info(f"Cohen's d (SCD: simple vs complex): {cohens_d_simple_complex:.4f}")

results['cohens_d_error_diff'] = float(cohens_d_errors)
results['cohens_d_scd_simple_complex'] = float(cohens_d_simple_complex)

## Metric 8: Normality Tests

Tests whether error distributions are normally distributed using Shapiro-Wilk test.
Non-normal errors may require non-parametric tests.

In [ ]:
# METRIC 8: NORMALITY TESTS
logger.info("=" * 60)
logger.info("METRIC 8: NORMALITY TESTS (Shapiro-Wilk on error distributions)")
logger.info("=" * 60)

w_scd, p_w_scd = shapiro(scd_errors)
w_fk, p_w_fk = shapiro(fk_errors)

logger.info(f"SCD errors: W = {w_scd:.4f}, p = {p_w_scd:.6f}")
logger.info(f"  {'Normal distribution' if p_w_scd >= 0.05 else 'Non-normal distribution'}")

logger.info(f"FK errors: W = {w_fk:.4f}, p = {p_w_fk:.6f}")
logger.info(f"  {'Normal distribution' if p_w_fk >= 0.05 else 'Non-normal distribution'}")

results['normality_scd_errors_w'] = float(w_scd)
results['normality_scd_errors_p'] = float(p_w_scd)
results['normality_fk_errors_w'] = float(w_fk)
results['normality_fk_errors_p'] = float(p_w_fk)

## Additional Analyses

Additional statistical metrics:
- Spearman rank correlation (non-parametric)
- R-squared (variance explained)

In [ ]:
# ADDITIONAL ANALYSES
logger.info("=" * 60)
logger.info("ADDITIONAL ANALYSES")
logger.info("=" * 60)

# Spearman rank correlation (non-parametric)
rho_scd, p_rho_scd = stats.spearmanr(scd_scores, true_grades)
rho_fk, p_rho_fk = stats.spearmanr(fk_scores, true_grades)

logger.info(f"Spearman correlation (SCD vs true): rho = {rho_scd:.4f}, p = {p_rho_scd:.6f}")
logger.info(f"Spearman correlation (FK vs true): rho = {rho_fk:.4f}, p = {p_rho_fk:.6f}")

results['spearman_scd_true'] = float(rho_scd)
results['spearman_fk_true'] = float(rho_fk)

# R-squared
r2_scd = r_scd ** 2
r2_fk = r_fk ** 2

logger.info(f"R-squared (SCD): {r2_scd:.4f} ({100*r2_scd:.1f}% of variance explained)")
logger.info(f"R-squared (FK): {r2_fk:.4f} ({100*r2_fk:.1f}% of variance explained)")

results['r2_scd'] = float(r2_scd)
results['r2_fk'] = float(r2_fk)

## Summary of Results

Prints a comprehensive summary of all evaluation metrics.

In [ ]:
# SUMMARY
logger.info("=" * 60)
logger.info("SUMMARY OF RESULTS")
logger.info("=" * 60)

logger.info(f"1. Correlation with true grades:")
logger.info(f"   SCD: r = {r_scd:.4f} [{ci_low_scd:.4f}, {ci_high_scd:.4f}]")
logger.info(f"   FK:  r = {r_fk:.4f} [{ci_low_fk:.4f}, {ci_high_fk:.4f}]")
logger.info(f"   Williams test: t = {t_williams:.4f}, p = {p_williams:.6f}")

logger.info(f"2. Error rates:")
logger.info(f"   SCD: MAE = {np.mean(scd_errors):.4f}, RMSE = {rmse_scd:.4f}")
logger.info(f"   FK:  MAE = {np.mean(fk_errors):.4f}, RMSE = {rmse_fk:.4f}")

logger.info(f"3. Complementarity:")
logger.info(f"   SCD-FK correlation: r = {r_scd_fk:.4f}")
logger.info(f"   Partial correlation (SCD|FK): r = {r_partial_scd:.4f}, p = {p_partial_scd:.6f}")
logger.info(f"   Ensemble correlation: r = {r_ensemble:.4f}")

logger.info(f"4. Computational efficiency:")
logger.info(f"   SCD: {mean_scd_time:.4f} ms/text")
logger.info(f"   FK:  {mean_fk_time:.6f} ms/text")

logger.info(f"5. ANOVA (SCD across complexity): F = {f_stat:.4f}, p = {p_anova:.6f}")

## Visualization

Creates plots to visualize the evaluation results:
1. Scatter plots of predicted vs true grades
2. Error distribution comparison
3. Bar chart of key metrics

In [ ]:
# VISUALIZATION - Key results with matplotlib
import matplotlib.pyplot as plt

# Set up the figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('SCD Readability Metric Evaluation Results', fontsize=16)

# Plot 1: Scatter plot SCD vs True Grades
axes[0, 0].scatter(true_grades, scd_scores, alpha=0.6, color='blue')
axes[0, 0].set_xlabel('True Grade Level')
axes[0, 0].set_ylabel('SCD Score')
axes[0, 0].set_title(f'SCD vs True (r={r_scd:.3f})')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Scatter plot FK vs True Grades
axes[0, 1].scatter(true_grades, fk_scores, alpha=0.6, color='green')
axes[0, 1].set_xlabel('True Grade Level')
axes[0, 1].set_ylabel('FK Score')
axes[0, 1].set_title(f'FK vs True (r={r_fk:.3f})')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Error distribution comparison
axes[0, 2].boxplot([scd_errors, fk_errors], labels=['SCD', 'FK'])
axes[0, 2].set_ylabel('Absolute Error')
axes[0, 2].set_title(f'Error Distribution (MAE: SCD={mae_scd:.2f}, FK={mae_fk:.2f})')
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: SCD scores by complexity level
complexity_levels = ['simple', 'medium', 'complex']
scd_by_complexity = [simple_scd, medium_scd, complex_scd]
axes[1, 0].boxplot(scd_by_complexity, labels=complexity_levels)
axes[1, 0].set_ylabel('SCD Score')
axes[1, 0].set_title(f'SCD by Complexity (F={f_stat:.1f}, p={p_anova:.3f})')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Bar chart of correlations
correlations = [r_scd, r_fk, r_ensemble, r_partial_scd]
corr_labels = ['SCD', 'FK', 'Ensemble', 'SCD|FK']
axes[1, 1].bar(corr_labels, correlations, color=['blue', 'green', 'purple', 'orange'])
axes[1, 1].set_ylabel('Pearson Correlation')
axes[1, 1].set_title('Correlation with True Grades')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(correlations):
    axes[1, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

# Plot 6: Summary table as text
axes[1, 2].axis('off')
summary_text = (
    f"Key Results Summary:\n"
    f"===================\n\n"
    f"Correlation with True Grades:\n"
    f"  SCD: r = {r_scd:.3f} [{ci_low_scd:.3f}, {ci_high_scd:.3f}]\n"
    f"  FK:  r = {r_fk:.3f} [{ci_low_fk:.3f}, {ci_high_fk:.3f}]\n"
    f"  Williams test: p = {p_williams:.3f}\n\n"
    f"Error Rates:\n"
    f"  SCD MAE = {mae_scd:.2f}\n"
    f"  FK MAE = {mae_fk:.2f}\n\n"
    f"Complementarity:\n"
    f"  SCD-FK correlation: r = {r_scd_fk:.3f}\n"
    f"  Ensemble: r = {r_ensemble:.3f}\n\n"
    f"ANOVA (SCD across complexity):\n"
    f"  F = {f_stat:.1f}, p = {p_anova:.2e}\n\n"
    f"Cohen's d (error diff): {cohens_d_errors:.2f}"
)
axes[1, 2].text(0.1, 0.9, summary_text, fontsize=10, family='monospace',
               transform=axes[1, 2].transAxes, verticalalignment='top')

plt.tight_layout()
plt.show()

# Print results as a table
print("\n" + "="*80)
print("EVALUATION RESULTS TABLE")
print("="*80)
print(f"{'Metric':<40} {'SCD':<15} {'FK':<15} {'Notes'}")
print("-"*80)
print(f"{'Pearson Correlation':<40} {r_scd:<15.4f} {r_fk:<15.4f} Williams p={p_williams:.3f}")
print(f"{'95% CI Lower':<40} {ci_low_scd:<15.4f} {ci_low_fk:<15.4f}")
print(f"{'95% CI Upper':<40} {ci_high_scd:<15.4f} {ci_high_fk:<15.4f}")
print(f"{'MAE':<40} {mae_scd:<15.4f} {mae_fk:<15.4f}")
print(f"{'RMSE':<40} {rmse_scd:<15.4f} {rmse_fk:<15.4f}")
print(f"{'ANOVA F-stat':<40} {f_stat:<15.4f} {f_stat_fk:<15.4f}")
print(f"{'Ensemble Correlation':<40} {r_ensemble:<15.4f} {'':<15} SCD+FK")
print(f"{'Partial Corr (SCD|FK)':<40} {r_partial_scd:<15.4f} {'':<15} p={p_partial_scd:.3f}")
print("="*80)

In [ ]:
# Save results to JSON (optional)
output_results = {
    'metadata': {
        'evaluation_name': 'SCD Readability Metric Statistical Evaluation (Demo)',
        'description': 'Demo evaluation with mini dataset',
        'n_examples': len(df),
        'complexity_levels': df['complexity'].value_counts().to_dict(),
    },
    'metrics_agg': results,
}

with open('demo_eval_results.json', 'w') as f:
    json.dump(output_results, f, indent=2)

print("Results saved to demo_eval_results.json")
print("\nDemo evaluation complete!")
print("\nTo run with full data:")
print("  1. Set N_BOOTSTRAP = 2000 in the Configuration cell")
print("  2. Load full_method_out.json instead of mini_demo_data.json")